# Unrestricted Image Generator (NEW)

Safety checker OFF · Adult OK · Adults only

### Steps
1. **Runtime → Change runtime type → GPU** (A100 / T4 / L4) → **Save**
2. Neeche wala cell **Run** karo
3. `https://xxxx.gradio.live` link open karo
4. Prompt likho → **Generate**

Pehli run ~6GB download karegi. Purani Untitled notebooks mat chalao.

In [ ]:
# ===== RUN (GPU pehle ON karo) =====
import os, sys
from pathlib import Path

import torch
assert torch.cuda.is_available(), "GPU OFF hai! Runtime → Change runtime type → GPU (A100/T4/L4) → Save"

print("GPU:", torch.cuda.get_device_name(0))

%pip install -q "diffusers==0.29.2" "transformers==4.41.2" "accelerate==0.30.1" "gradio==4.44.1" safetensors sentencepiece pillow

root = Path("/content") if Path("/content").exists() else Path.cwd()
(root / "app.py").write_text('#!/usr/bin/env python3\n"""Unrestricted text-to-image + img2img. Safety checker OFF. Adults only."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport gc\nimport inspect\nimport os\n\nimport gradio as gr\nimport torch\nfrom diffusers import (\n    StableDiffusionXLImg2ImgPipeline,\n    StableDiffusionXLPipeline,\n)\nfrom PIL import Image\n\nMODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"\n# NSFW-friendly alternative (uncomment to use):\n# MODEL_ID = "John6666/pony-realism-v21main-sdxl"\n\nNEG = (\n    "blurry, low quality, worst quality, deformed, ugly, bad anatomy, "\n    "extra limbs, missing fingers, watermark, text, child, underage, loli"\n)\n\npipe = None\npipe_i2i = None\n\n\ndef _device() -> str:\n    return "cuda" if torch.cuda.is_available() else "cpu"\n\n\ndef _gpu_label() -> str:\n    if not torch.cuda.is_available():\n        return "No GPU"\n    name = torch.cuda.get_device_name(0)\n    gb = torch.cuda.get_device_properties(0).total_memory / 1024**3\n    return f"{name} ({gb:.0f} GB)"\n\n\ndef _from_components(cls, base):\n    params = inspect.signature(cls.__init__).parameters\n    kw = {k: v for k, v in base.components.items() if k in params and v is not None}\n    return cls(**kw)\n\n\ndef load_model():\n    global pipe, pipe_i2i\n    if _device() != "cuda":\n        raise RuntimeError(\n            "GPU chahiye. Colab: Runtime → Change runtime type → GPU (A100/T4/L4) → Save"\n        )\n\n    print(f"GPU: {_gpu_label()}")\n    print(f"Loading {MODEL_ID} … (pehli dafa download hoga)")\n\n    pipe = None\n    pipe_i2i = None\n    gc.collect()\n    torch.cuda.empty_cache()\n\n    kwargs = dict(torch_dtype=torch.float16, use_safetensors=True)\n    try:\n        p = StableDiffusionXLPipeline.from_pretrained(\n            MODEL_ID, variant="fp16", add_watermarker=False, **kwargs\n        )\n    except TypeError:\n        try:\n            p = StableDiffusionXLPipeline.from_pretrained(\n                MODEL_ID, variant="fp16", **kwargs\n            )\n        except Exception:\n            p = StableDiffusionXLPipeline.from_pretrained(MODEL_ID, **kwargs)\n    except Exception:\n        p = StableDiffusionXLPipeline.from_pretrained(MODEL_ID, **kwargs)\n\n    p = p.to("cuda")\n    p.enable_attention_slicing()\n    try:\n        p.enable_vae_slicing()\n    except Exception:\n        pass\n\n    pipe = p\n    pipe_i2i = _from_components(StableDiffusionXLImg2ImgPipeline, p)\n    pipe_i2i.enable_attention_slicing()\n\n    used = torch.cuda.memory_allocated() / 1024**3\n    msg = f"Ready — {_gpu_label()} — model {used:.1f} GB"\n    print(msg)\n    return msg\n\n\ndef generate(prompt, negative, steps, guidance, width, height, seed):\n    if pipe is None:\n        raise gr.Error("Model load nahi hua. Cell dubara run karo.")\n    if not (prompt or "").strip():\n        raise gr.Error("Prompt likho.")\n\n    w = max(768, min(1536, (int(width) // 64) * 64))\n    h = max(768, min(1536, (int(height) // 64) * 64))\n    gen = None\n    if seed is not None and int(seed) >= 0:\n        gen = torch.Generator("cuda").manual_seed(int(seed))\n\n    out = pipe(\n        prompt=prompt,\n        negative_prompt=negative or "",\n        num_inference_steps=int(steps),\n        guidance_scale=float(guidance),\n        width=w,\n        height=h,\n        generator=gen,\n    )\n    return out.images[0]\n\n\ndef edit(image, prompt, negative, strength, steps, guidance, seed):\n    if pipe_i2i is None:\n        raise gr.Error("Model load nahi hua.")\n    if image is None:\n        raise gr.Error("Image upload karo.")\n    if not (prompt or "").strip():\n        raise gr.Error("Edit prompt likho.")\n\n    gen = None\n    if seed is not None and int(seed) >= 0:\n        gen = torch.Generator("cuda").manual_seed(int(seed))\n\n    out = pipe_i2i(\n        prompt=prompt,\n        negative_prompt=negative or "",\n        image=image.convert("RGB"),\n        strength=float(strength),\n        num_inference_steps=int(steps),\n        guidance_scale=float(guidance),\n        generator=gen,\n    )\n    return out.images[0]\n\n\ndef build_ui(status: str = "") -> gr.Blocks:\n    with gr.Blocks(title="Unrestricted Image Generator") as demo:\n        gr.Markdown(\n            f"""\n# Unrestricted Image Generator\nSafety checker **OFF** · Adult OK · **Adults only** (no minors)\n\n**GPU:** {_gpu_label()}\n            """\n        )\n        gr.Textbox(value=status or "Model ready", label="Status", interactive=False)\n\n        with gr.Tabs():\n            with gr.Tab("Generate"):\n                prompt = gr.Textbox(label="Prompt", lines=4, placeholder="Detailed prompt…")\n                neg = gr.Textbox(label="Negative", value=NEG, lines=2)\n                with gr.Row():\n                    steps = gr.Slider(15, 50, value=28, step=1, label="Steps")\n                    guidance = gr.Slider(1, 12, value=7, step=0.5, label="Guidance")\n                with gr.Row():\n                    width = gr.Slider(768, 1536, value=1024, step=64, label="Width")\n                    height = gr.Slider(768, 1536, value=1024, step=64, label="Height")\n                    seed = gr.Number(value=-1, label="Seed (-1 random)", precision=0)\n                btn = gr.Button("Generate", variant="primary")\n                out = gr.Image(label="Result", height=512)\n                btn.click(generate, [prompt, neg, steps, guidance, width, height, seed], out)\n\n            with gr.Tab("Edit"):\n                img = gr.Image(type="pil", label="Upload", height=320)\n                prompt_e = gr.Textbox(label="Edit prompt", lines=3)\n                neg_e = gr.Textbox(label="Negative", value=NEG, lines=2)\n                with gr.Row():\n                    strength = gr.Slider(0.2, 0.95, value=0.55, step=0.05, label="Strength")\n                    steps_e = gr.Slider(15, 50, value=28, step=1, label="Steps")\n                    guidance_e = gr.Slider(1, 12, value=7, step=0.5, label="Guidance")\n                seed_e = gr.Number(value=-1, label="Seed (-1 random)", precision=0)\n                btn_e = gr.Button("Edit", variant="primary")\n                out_e = gr.Image(label="Result", height=512)\n                btn_e.click(\n                    edit, [img, prompt_e, neg_e, strength, steps_e, guidance_e, seed_e], out_e\n                )\n    return demo\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--share", action="store_true")\n    parser.add_argument("--port", type=int, default=7860)\n    parser.add_argument("--host", default="0.0.0.0")\n    args = parser.parse_args()\n\n    os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")\n    status = load_model()\n    demo = build_ui(status)\n    demo.queue().launch(\n        server_name=args.host,\n        server_port=args.port,\n        share=args.share,\n        show_error=True,\n    )\n\n\nif __name__ == "__main__":\n    main()\n')
sys.path.insert(0, str(root))

os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

import importlib
import app as gen
importlib.reload(gen)

status = gen.load_model()
demo = gen.build_ui(status)
demo.queue().launch(share=True, server_name="0.0.0.0", show_error=True)
